# shard-prompt — chỉ pha retrieval trên tập private

Không nạp Qwen, không sinh đáp án, không nộp LB. Mỗi shard ghi `prompts.jsonl`
cùng schema với `prompts_g2` (id/question/prompt/doc_ids) rồi `ghi_moc` khi đủ
`[tu, den)`. Dense + rerank khớp đường G15: AITeamVN + `rerank_v11`.

In [ ]:
# ORCHESTRA-PARAMS — kaggle_orchestra.py thay cell này lúc đẩy. Giá trị dưới chỉ để chạy tay.
SHARD = {"job": "tay", "chi_so": 0, "so_shard": 1, "tu": 0, "den": 4, "commit": "", "noi_tiep": None, "output": "prompts.jsonl"}

In [ ]:
import os, sys, shutil, subprocess, json
from pathlib import Path

SRC = next((c.parent.parent.parent for c in Path("/kaggle/input").glob("**/legalqa/bm25.py")
            if "legalqa-v10-src" in str(c)), None)
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/tmp/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
commit = (REPO / "COMMIT.txt").read_text().strip()
print("codebase:", commit, "| shard", SHARD)
assert not SHARD["commit"] or commit == SHARD["commit"], f"src {commit} != {SHARD['commit']}"

OUT = Path("/kaggle/working/out"); OUT.mkdir(parents=True, exist_ok=True)
if SHARD["noi_tiep"]:
    for f in Path("/kaggle/input").glob("**/*.jsonl"):
        if SHARD["noi_tiep"] in str(f):
            shutil.copy(f, OUT / f.name)
            print(f"nối tiếp {f.name}: {sum(1 for _ in open(OUT / f.name, encoding='utf-8')):,} dòng")

In [ ]:
!pip -q install transformers==5.14.1
!pip -q uninstall -y torchao

In [ ]:
os.environ["LEGALQA_SO_HIEU"] = "1"

ch = next((p for p in Path("/kaggle/input").glob("**/chunks.jsonl")), None)
assert ch, "Không thấy chunks.jsonl — gắn dataset legalqa-clean"
CLEAN = Path("/tmp/clean"); CLEAN.mkdir(parents=True, exist_ok=True)
for f in ch.parent.iterdir():
    if not (CLEAN / f.name).exists():
        (CLEAN / f.name).symlink_to(f)
# Cache dense: không encode lại 287k chunk trên mỗi shard. rerank_v11 finetune trên pool
# RRF của HAI nhánh dense (mine_rerank.py mặc định e5-small + VN-Embedding) — pool 1-dense
# lệch phân phối với lúc finetune, nên symlink cả hai cache ở đây.
for stem in ("dense_Vietnamese_Embedding_L256", "dense_multilingual-e5-small_L256"):
    for npy in Path("/kaggle/input").glob(f"**/{stem}.npy"):
        if not (CLEAN / npy.name).exists():
            (CLEAN / npy.name).symlink_to(npy)
        ids = npy.with_name(f"{stem}.ids")
        if ids.exists() and not (CLEAN / ids.name).exists():
            (CLEAN / ids.name).symlink_to(ids)
# CLEAN_DIR phải set TRƯỚC import build_sft: module gán CLEAN lúc import.
os.environ["CLEAN_DIR"] = str(CLEAN)

from legalqa.prompt import CO_SO_HIEU
from legalqa.rerank import CrossEncoderReranker
from legalqa.retrieve import Retriever
from build_sft import TOP_N, MAX_CHARS, nap_chunks, nap_dense, nap_docs, nap_index
from infer import da_co, nap_cau_hoi, pha_prompt
from kaggle_orchestra import ghi_moc

assert CO_SO_HIEU is True and TOP_N == 5 and MAX_CHARS > 10**6

CAU_HOI = next(Path("/kaggle/input").glob("**/private_q.jsonl"), None)
assert CAU_HOI, "Không thấy private_q.jsonl — gắn dataset legalqa-private-q"
hang = nap_cau_hoi(CAU_HOI)
assert len(hang) == 1918, f"private có {len(hang)} câu, phải là 1.918 (cùng prompts_g2)"
rows = hang[SHARD["tu"]: SHARD["den"]]
print(f"{len(rows)} câu shard [{SHARD['tu']},{SHARD['den']})", flush=True)

local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text(encoding="utf-8")]
assert local, "Không thấy rerank_v11 — gắn dataset legalqa-rerank-v11"
RERANK = str(next((p for p in local if "rerank" in str(p).lower()), local[0]))
# THỨ TỰ khớp mine_rerank.py mặc định: e5-small trước, VN-Embedding sau (RRF 2 nhánh).
DENSE = SHARD.get("dense", "intfloat/multilingual-e5-small,AITeamVN/Vietnamese_Embedding")
ten_dense = [t for t in DENSE.split(",") if t]
assert len(ten_dense) == 2, f"cần đúng 2 model dense (RRF hybrid), nhận {ten_dense}"
print("rerank", RERANK, "| dense", ten_dense, flush=True)

reranker = CrossEncoderReranker(RERANK, batch_size=int(SHARD.get("batch_size", 32)))
chunks = nap_chunks()
encoder, dense_idx = nap_dense(ten_dense[0], chunks)
encoder2, dense_idx2 = nap_dense(ten_dense[1], chunks)
r = Retriever(index=nap_index(chunks), chunks=chunks, docs=nap_docs(),
              reranker=reranker, encoder=encoder, dense=dense_idx,
              encoder2=encoder2, dense2=dense_idx2,
              pool=int(SHARD.get("pool", 50)))

ra = OUT / SHARD["output"]
pha_prompt(rows, r, ra, float(SHARD.get("gio_pha1", 2.5)) * 3600)
can = {x["id"] for x in rows}
co = can & da_co(ra)
if co == can:
    print("mốc:", ghi_moc(ra, SHARD["tu"], SHARD["den"]), flush=True)
else:
    print(f"CHƯA XONG shard: {len(co)}/{len(can)} câu — không ghi mốc", flush=True)